# Facial Emotion CNN — Kaggle GPU training

Self-contained notebook (no dependency on cloning the repo) that trains a
CNN on FER2013 using CLAHE + grayscale + normalization preprocessing and
inline augmentation. Attach any FER2013 image-folder dataset via
'Add Input' (e.g. `astraszab/facial-expression-dataset-image-folders-fer2013`)
and enable GPU (T4) before running. The dataset cell below auto-detects
the train/test folder paths regardless of the exact dataset slug or
folder-naming convention, so no manual path editing should be needed.

Outputs land in `/kaggle/working/artifacts/` — download `model.keras`,
`history.json`, `metrics.json`, `confusion_matrix.png`, and
`training_curves.png` into this repo's `artifacts/` folder afterward.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print('GPUs:', tf.config.list_physical_devices('GPU'))

## Constants + preprocessing (CLAHE / grayscale / normalize)

In [ ]:
IMG_SIZE = 48
NUM_CLASSES = 7
EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


def apply_clahe(gray_image, clip_limit=2.0, tile_grid_size=8):
    if gray_image.dtype != np.uint8:
        gray_image = np.clip(gray_image, 0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid_size, tile_grid_size))
    return clahe.apply(gray_image)


def normalize(gray_image):
    return gray_image.astype(np.float32) / 255.0


def clahe_normalize_batch(images):
    images = images.numpy() if hasattr(images, 'numpy') else images
    processed = np.empty_like(images, dtype=np.float32)
    for i in range(images.shape[0]):
        gray = images[i, ..., 0].astype(np.uint8)
        gray = apply_clahe(gray)
        processed[i, ..., 0] = normalize(gray)
    return processed


def preprocess_batch(images, labels, use_clahe=True):
    if use_clahe:
        images = tf.py_function(clahe_normalize_batch, [images], tf.float32)
    else:
        images = tf.cast(images, tf.float32) / 255.0
    images.set_shape([None, IMG_SIZE, IMG_SIZE, 1])
    return images, labels

## Dataset — auto-detect the attached FER2013 layout

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
print('Attached inputs:', [p.name for p in INPUT_ROOT.iterdir()])

EMOTION_SET = {e.lower() for e in EMOTION_LABELS}


def find_split_folders(root: Path):
    """Find every directory whose immediate subfolders cover the 7 emotion
    classes, anywhere under `root`. Works regardless of dataset slug or
    nesting depth."""
    hits = []
    for p in root.rglob('*'):
        if not p.is_dir():
            continue
        child_names = {c.name.lower() for c in p.iterdir() if c.is_dir()}
        if EMOTION_SET.issubset(child_names):
            n_images = sum(1 for c in p.rglob('*') if c.is_file())
            hits.append((p, n_images))
    return hits


hits = find_split_folders(INPUT_ROOT)
hits.sort(key=lambda h: -h[1])
print('Candidate split folders (path, #image files):')
for p, n in hits:
    print(' ', p, n)

assert hits, (
    f'Could not find a folder with subfolders {sorted(EMOTION_SET)} under {INPUT_ROOT}. '
    'Make sure a FER2013 image-folder dataset is attached via Add Input.'
)


def pick(keyword_groups, exclude=None):
    for keywords in keyword_groups:
        for p, n in hits:
            if exclude is not None and p == exclude:
                continue
            if any(k in p.name.lower() for k in keywords):
                return p
    return None


train_dir = pick([['train', 'training']]) or hits[0][0]
test_dir = pick([['privatetest'], ['publictest'], ['test', 'testing', 'val', 'validation']], exclude=train_dir)
if test_dir is None:
    remaining = [h for h in hits if h[0] != train_dir]
    test_dir = remaining[0][0] if remaining else train_dir

print('Using train_dir =', train_dir)
print('Using test_dir  =', test_dir)

In [ ]:
OUT_DIR = Path('/kaggle/working/artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 60
VAL_SPLIT = 0.1
SEED = 42
USE_CLAHE = True
AUTOTUNE = tf.data.AUTOTUNE

common = dict(image_size=(IMG_SIZE, IMG_SIZE), color_mode='grayscale', batch_size=BATCH_SIZE, label_mode='int')

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=VAL_SPLIT, subset='training', seed=SEED, **common
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=VAL_SPLIT, subset='validation', seed=SEED, **common
)
test_ds = tf.keras.utils.image_dataset_from_directory(test_dir, shuffle=False, **common)

class_names = train_ds.class_names
print('Classes:', class_names)
assert EMOTION_SET.issubset({c.lower() for c in class_names}), f'Unexpected class folders: {class_names}'
EMOTION_LABELS = class_names  # use whichever order Keras discovered from the folders


def prep(ds):
    ds = ds.map(lambda x, y: preprocess_batch(x, y, USE_CLAHE), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds, val_ds, test_ds = prep(train_ds), prep(val_ds), prep(test_ds)

## Model (Conv2D / BatchNorm / MaxPooling / Dropout blocks + augmentation)

In [ ]:
def build_augmentation_layer():
    return tf.keras.Sequential(
        [
            layers.RandomRotation(0.08),
            layers.RandomTranslation(0.08, 0.08),
            layers.RandomZoom(0.1),
            layers.RandomFlip('horizontal'),
        ],
        name='augmentation',
    )


def conv_block(x, filters, dropout):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout)(x)
    return x


def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=NUM_CLASSES, use_augmentation=True):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    if use_augmentation:
        x = build_augmentation_layer()(x)
    x = conv_block(x, 32, 0.25)
    x = conv_block(x, 64, 0.25)
    x = conv_block(x, 128, 0.30)
    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='facial_emotion_cnn')


model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

with open(OUT_DIR / 'history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

In [ ]:
model.save(OUT_DIR / 'model.keras')

In [ ]:
# Real held-out evaluation on the FER2013 test split
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_true.extend(labels.numpy().tolist())

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
cm = confusion_matrix(y_true, y_pred)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

metrics = {
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'classification_report': report,
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
    'epochs_trained': len(history.history['loss']),
    'batch_size': BATCH_SIZE,
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('FER2013 test confusion matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
fig.colorbar(im)
fig.tight_layout()
fig.savefig(OUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'training_curves.png', dpi=150)
plt.show()